In [3]:
import json
import pickle
from pathlib import Path

import numpy as np

import sys
sys.path.append("../workflow/scripts")
from helpers import fit_clorinn, ensure_parent, clorinn_to_dict
from fit_cv_model import load_cv_data, heldout_metrics

In [4]:
cv_input = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cv_input/zscore_cv.npz"
method = "nnm"
nucnorm = float("8192")
max_iter = 1000
svd_max_iter = 100
model_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/models/nnm_cv_model_r128.pkl"
metrics_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/metrics/nnm_cv_metrics_r128.json"

In [5]:
ztrue, ztrain, zmask = load_cv_data(cv_input)
ztrain_filled = np.nan_to_num(ztrue, 0)
np.linalg.norm(ztrain_filled, ord = 'nuc')

29849.55340914954

In [6]:
from clorinn.optimize import PGDWarmStart

nan_mask = np.isnan(ztrain)
Y = np.nan_to_num(ztrain, nan=0.0)

# Phase 1: PGD
pgd = PGDWarmStart(max_iter=50, show_progress=True)
pgd.fit(Y, nucnorm, mask=nan_mask)
X0 = pgd.X
print(f"PGD finished in {pgd.n_iter} iterations")

2026-04-16 15:28:23,640 | optimize.simplex_projection | INFO    | PGD iter    1  f = 6323690.4558  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:24,186 | optimize.simplex_projection | INFO    | PGD iter    2  f = 2227191.2159  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:24,730 | optimize.simplex_projection | INFO    | PGD iter    3  f = 2200454.2497  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:25,274 | optimize.simplex_projection | INFO    | PGD iter    4  f = 2196734.3490  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:25,817 | optimize.simplex_projection | INFO    | PGD iter    5  f = 2195968.9558  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:26,361 | optimize.simplex_projection | INFO    | PGD iter    6  f = 2195783.2574  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:26,905 | optimize.simplex_projection | INFO    | PGD iter    7  f = 2195733.5048  ||X||_* = 8192.0  (clipped)
2026-04-16 15:28:27,451 | optimize.simplex_projection | INFO    | PGD iter    8  f = 2195719.1456  ||X||_* = 81

In [11]:
np.linalg.norm(X0, ord='nuc')

8192.000000000004

In [9]:
from clorinn import FrankWolfe, MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(X0)
s2 = mf.s**2
np.cumsum(s2)/np.sum(s2)

array([0.22026024, 0.42796969, 0.56126262, 0.63703999, 0.70420253,
       0.76154791, 0.81025458, 0.85233738, 0.88385136, 0.90652697,
       0.92602258, 0.94015811, 0.95180692, 0.96300789, 0.97044551,
       0.97691343, 0.98188209, 0.98618819, 0.98947359, 0.99221727,
       0.99479611, 0.99653723, 0.99770036, 0.99861757, 0.99898768,
       0.99930661, 0.99951584, 0.99969044, 0.99979172, 0.99987608,
       0.99995801, 0.99998627, 0.9999981 , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.     

In [10]:
from clorinn import FrankWolfe
nnm_away = FrankWolfe(
    model='nnm', max_iter=1000, svd_method='left-gram',
    stop_criteria=['duality_gap'], tol=1e-2,
    show_progress=True, away_step=True, print_skip = 100
)
nnm_away.fit(ztrain, 8192, X0 = X0)

2026-04-16 15:33:52,689 | optimize.frankwolfe      | INFO    | Iteration 1. Step size 0.000. Duality Gap 630.399. Active set 35
2026-04-16 15:34:14,765 | optimize.frankwolfe      | INFO    | Iteration 100. Step size 0.000. Duality Gap 40.1263. Active set 95
2026-04-16 15:34:38,055 | optimize.frankwolfe      | INFO    | Iteration 200. Step size 0.000. Duality Gap 20.7599. Active set 161
2026-04-16 15:35:02,462 | optimize.frankwolfe      | INFO    | Iteration 300. Step size 0.000. Duality Gap 15.2869. Active set 227
2026-04-16 15:35:27,841 | optimize.frankwolfe      | INFO    | Iteration 400. Step size 0.000. Duality Gap 11.3982. Active set 293
2026-04-16 15:35:54,252 | optimize.frankwolfe      | INFO    | Iteration 500. Step size 0.000. Duality Gap 7.66983. Active set 353
2026-04-16 15:36:21,621 | optimize.frankwolfe      | INFO    | Iteration 600. Step size 0.000. Duality Gap 8.60988. Active set 422
2026-04-16 15:36:50,122 | optimize.frankwolfe      | INFO    | Iteration 700. Step size

In [12]:
np.sum(np.linalg.svd(nnm_away.X, compute_uv=False))

8191.9999998967905

In [35]:
from clorinn import FrankWolfe
nnm = FrankWolfe(
    model='nnm', max_iter=1000, svd_method='left-gram',
    stop_criteria=['duality_gap'], tol=1e-2,
    show_progress=True, away_step=False, print_skip = 100
)
nnm.fit(ztrain, 8192, X0 = X0)

2026-04-14 17:29:55,324 | optimize.frankwolfe      | INFO    | Iteration 1. Step size 0.000. Duality Gap 1.28482e-08
2026-04-14 17:29:55,325 | optimize.frankwolfe      | INFO    | Stopping at iteration 1. Step size 0.000. Duality Gap 1.28482e-08
2026-04-14 17:29:55,325 | optimize.frankwolfe      | INFO    | Duality gap converged below tolerance.


In [36]:
np.sum(np.linalg.svd(nnm.X, compute_uv=False))

8191.999999999995

In [13]:
from clorinn import FrankWolfe, MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(X0)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("X0 after PGD")
print (energy[:20])

mf = MatrixFactorization(k = 20)
mf.fit(nnm_away.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("Away Step FW after PGD")
print (energy[:20])

X0 after PGD
[0.22026024 0.42796969 0.56126262 0.63703999 0.70420253 0.76154791
 0.81025458 0.85233738 0.88385136 0.90652697 0.92602258 0.94015811
 0.95180692 0.96300789 0.97044551 0.97691343 0.98188209 0.98618819
 0.98947359 0.99221727]
Away Step FW after PGD
[0.22027905 0.42798405 0.56127691 0.63705748 0.70422012 0.7615641
 0.81027359 0.8523576  0.88387017 0.90654486 0.92603762 0.94017088
 0.95181716 0.96301664 0.97045305 0.97691931 0.98188732 0.98619211
 0.98947695 0.99222007]


In [15]:
print(f"PGD objective:         {pgd.fx[-1]:.2f}")
print(f"FW after 1000 steps:   {nnm_away.fx[-1]:.2f}")
print(f"Relative improvement:  {(pgd.fx[-1] - nnm_away.fx[-1]) / pgd.fx[-1]:g}")

PGD objective:         2195713.31
FW after 1000 steps:   2195712.81
Relative improvement:  2.26413e-07


In [26]:
mf = MatrixFactorization(k = 20)
mf.fit(nnm.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print (energy[:20])

[0.2202798  0.42798556 0.56127875 0.63705931 0.70422185 0.76156574
 0.81027512 0.8523589  0.88387131 0.90654572 0.92603829 0.94017134
 0.95181748 0.96301678 0.97045305 0.9769192  0.98188709 0.98619183
 0.98947657 0.99221962]


[0.22026024 0.42796969 0.56126262 0.63703999 0.70420253 0.76154791
 0.81025458 0.85233738 0.88385136 0.90652697 0.92602258 0.94015811
 0.95180692 0.96300789 0.97044551 0.97691343 0.98188209 0.98618819
 0.98947359 0.99221727]


In [10]:
model = fit_clorinn(ztrain, method, nucnorm, max_iter=max_iter, svd_method = 'left-gram', stop_criteria=['duality_gap'], print_skip = 100)

2026-04-14 15:52:04,840 | optimize.frankwolfe      | INFO    | Iteration 1. Step size 0.155. Duality Gap 9.42664e+06
2026-04-14 15:52:22,604 | optimize.frankwolfe      | INFO    | Iteration 100. Step size 0.006. Duality Gap 397534
2026-04-14 15:52:40,481 | optimize.frankwolfe      | INFO    | Iteration 200. Step size 0.004. Duality Gap 232309
2026-04-14 15:52:58,402 | optimize.frankwolfe      | INFO    | Iteration 300. Step size 0.003. Duality Gap 160204
2026-04-14 15:53:16,285 | optimize.frankwolfe      | INFO    | Iteration 400. Step size 0.002. Duality Gap 123149
2026-04-14 15:53:34,166 | optimize.frankwolfe      | INFO    | Iteration 500. Step size 0.002. Duality Gap 97655.5
2026-04-14 15:53:52,094 | optimize.frankwolfe      | INFO    | Iteration 600. Step size 0.001. Duality Gap 87041
2026-04-14 15:54:10,041 | optimize.frankwolfe      | INFO    | Iteration 700. Step size 0.001. Duality Gap 74943.9
2026-04-14 15:54:28,002 | optimize.frankwolfe      | INFO    | Iteration 800. Step s

In [11]:
np.sum(np.linalg.svd(model.X, compute_uv=False))

8092.994791043332

In [11]:
model.st_list_

[1.0,
 4.518844324396195e-06,
 4.252030160627611e-06,
 2.3828654045556675e-06,
 2.600069665654911e-06,
 2.69330200631097e-06,
 3.0006578820756483e-06,
 2.2485868423836745e-06,
 1.9550073844290154e-06,
 1.8427223566303564e-06,
 1.7038940718717094e-06,
 1.798815954789042e-06,
 1.810281193398985e-06,
 1.7404041137861201e-06,
 1.5680764744155576e-06,
 1.6837380025781087e-06,
 1.4321239531453822e-06,
 1.5033944483727934e-06,
 1.5064179308752185e-06,
 1.3489409643769899e-06,
 1.3104161914485132e-06,
 1.3874127479090354e-06,
 1.1887078308794055e-06,
 1.0899177092433275e-06,
 1.1257544922270298e-06,
 1.1127383008242503e-06,
 1.11742577613958e-06,
 1.2165460116817885e-06,
 1.2040417525070435e-06,
 1.0570887693198674e-06,
 1.1619626023989431e-06,
 1.049024473431108e-06,
 9.714933585325906e-07,
 8.76103208801173e-07,
 9.540491482601017e-07,
 1.0310586058349198e-06,
 1.0325813645351595e-06,
 9.75163259258827e-07,
 1.0145738140661343e-06,
 9.091883311043167e-07,
 9.493095171976575e-07,
 8.066031010

In [29]:
model.dg_list_

[inf, 1.202130306410254e-08]

In [12]:
model.fx_list_

[2195712.8172998815,
 2195712.816739453,
 2195712.8162208446,
 2195712.816063198,
 2195712.8158783414,
 2195712.815692784,
 2195712.8154572113,
 2195712.8153176126,
 2195712.8152126065,
 2195712.8151198123,
 2195712.8150403854,
 2195712.8149559763,
 2195712.8148689554,
 2195712.8147827317,
 2195712.8147129314,
 2195712.8146371343,
 2195712.814577851,
 2195712.814516996,
 2195712.81445835,
 2195712.8144079796,
 2195712.814361144,
 2195712.814310235,
 2195712.8142711963,
 2195712.8142359243,
 2195712.814201383,
 2195712.814166876,
 2195712.8141325326,
 2195712.8140932107,
 2195712.8140536505,
 2195712.8140202924,
 2195712.813984846,
 2195712.8139545415,
 2195712.813928231,
 2195712.8139053164,
 2195712.8138801144,
 2195712.813851056,
 2195712.813822408,
 2195712.8137969025,
 2195712.8137688283,
 2195712.813745372,
 2195712.8137213443,
 2195712.8137018164,
 2195712.8136795894,
 2195712.813655373,
 2195712.8136363244,
 2195712.813617092,
 2195712.813596262,
 2195712.813570977,
 2195712.813

In [21]:
ztrain_filled = np.nan_to_num(ztrain, 0)
np.linalg.norm(nnm_away.X, ord = 'fro') / np.linalg.norm(ztrain, ord = 'fro')

0.5812904834840025

In [9]:
# -- Save results ------
metrics = heldout_metrics(ztrue, model.X, zmask)
metrics.update(
    {
        "method": method,
        "nucnorm": nucnorm,
        "max_iter": max_iter,
        "n_iter" : len(model.steps),
    }
)
model_dict = clorinn_to_dict(model)

ensure_parent(model_out)
ensure_parent(metrics_out)

with open(model_out, "wb") as handle:
    pickle.dump(model_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(metrics_out, "w") as handle:
    json.dump(metrics, handle, indent=2, sort_keys=True)

print(f"Finished model fit at nucnorm={nucnorm:g}")
print(f"Held-out MSE: {metrics['heldout_mse']:.10f}")

Finished model fit at nucnorm=256
Held-out MSE: 2.8656791800


In [ ]:
from sklearn.utils.extmath import randomized_svd
from clorinn.optimize import TopCompSVD

dg = (X0 - z0_pgd) * ~nan_mask

print (f"Any NaNs: {np.any(np.isnan(dg))}")
print (f"Any Infs: {np.any(np.isinf(dg))}")

u1 = {}
v1_t = {}
for method in ('power', 'direct'):
    svd = TopCompSVD(method = method, max_iter = 2000)
    svd.fit(dg)
    u1[method] = svd.u1
    v1_t[method] = svd.v1_t
    if method == 'power':
        print (f"Number of power iterations: {svd.n_iter}")
        
print (f"u1 norm diff: {np.linalg.norm(u1['power'] - u1['direct'], ord='fro')}")
print (f"v1 norm diff: {np.linalg.norm(v1_t['power'] - v1_t['direct'], ord='fro')}")

In [ ]:
def project_nuclear_norm_ball(X, r):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    if np.sum(s) <= r:
        return X, U, s, Vt
    s_proj = project_simplex(s, r)
    return (U * s_proj) @ Vt, U, s_proj, Vt


def project_simplex(s, r):
    n = len(s)
    s_sorted = np.sort(s)[::-1]
    cumsum = np.cumsum(s_sorted)
    rho = np.where(s_sorted > (cumsum - r) / np.arange(1, n + 1))[0][-1]
    lam = (cumsum[rho] - r) / (rho + 1)
    return np.maximum(s - lam, 0)


def pgd_warmstart(Y, r, mask, max_iter=50, rel_tol=1e-6):
    """
    Adaptive PGD warm start. Runs until either:
      - the nuclear norm constraint becomes active (hand off to FW), or
      - the objective stalls (converged, FW will confirm via duality gap)
    """
    obs = ~mask
    X = np.zeros_like(Y)
    fx_old = np.inf

    for t in range(max_iter):
        G = np.where(obs, X - Y, 0.0)
        fx = 0.5 * np.linalg.norm(G, 'fro')**2

        # Gradient step
        X_candidate = X - G
        
        # Project and check if clipping occurred
        U, s, Vt = np.linalg.svd(X_candidate, full_matrices=False)
        nuc_before = np.sum(s)
        
        if nuc_before > r:
            s = project_simplex(s, r)
            X = (U * s) @ Vt
            print(f"PGD iter {t:3d}  f = {fx:.2f}  ||X||_* clipped "
                  f"({nuc_before:.1f} -> {r:.1f}). Handing off to FW.")
            break
        else:
            X = X_candidate

        # Check for stalling (already near-optimal in the interior)
        rel = abs(fx - fx_old) / max(1.0, abs(fx_old))
        if t > 0 and rel < rel_tol:
            print(f"PGD iter {t:3d}  f = {fx:.2f}  converged in interior "
                  f"(||X||_* = {nuc_before:.1f} < r = {r:.1f})")
            break
        fx_old = fx

        if t % 10 == 0:
            print(f"PGD iter {t:3d}  f = {fx:.2f}  ||X||_* = {nuc_before:.1f}")

    return X